# Homework 4 

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load datasets
weather_data = pd.read_csv('BicycleWeather.csv')
fremont_data = pd.read_csv('FremontBridge.csv')

# Check the first few rows of the data to understand the format
print("Weather Data Sample:\n", weather_data.head())
print("Fremont Data Sample:\n", fremont_data.head())

# Convert 'DATE' columns in both datasets to datetime format with a custom format
fremont_data['DATE'] = pd.to_datetime(fremont_data['DATE'], errors='coerce', format='%m/%d/%Y %I:%M:%S %p')
weather_data['DATE'] = pd.to_datetime(weather_data['DATE'], errors='coerce', format='%Y%m%d')

# Check if there are any invalid or missing values after conversion
print("Fremont Data after Date Conversion:\n", fremont_data.head())
print("Weather Data after Date Conversion:\n", weather_data.head())

# Ensure there is no missing data
print("Missing data in Fremont Data:\n", fremont_data.isnull().sum())
print("Missing data in Weather Data:\n", weather_data.isnull().sum())

# Merge the datasets on 'DATE'
merged_data = pd.merge(weather_data, fremont_data, on='DATE', how='inner')

# Check if the merged dataset has rows
if merged_data.empty:
    print("The merged dataset is empty. Please check the date range and data formats.")
else:
    print("Merged Data Sample:\n", merged_data.head())

    # Handle missing data by filling with 0
    merged_data.fillna(0, inplace=True)

    # Create the 'cyclist_count' target by summing the East and West Sidewalk counts
    merged_data['cyclist_count'] = merged_data['Fremont Bridge East Sidewalk'] + merged_data['Fremont Bridge West Sidewalk']

    # Define the features (X) and target (y)
    X = merged_data[['PRCP', 'TMAX', 'TMIN', 'AWND']]  # Example features
    y = merged_data['cyclist_count']

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features for models requiring normalization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define models with cross-validation for alpha selection
    models = {
        'LinearRegression': LinearRegression(),
        'Lasso': LassoCV(cv=10),
        'Ridge': RidgeCV(cv=10)
    }

    # Perform cross-validation and fit models
    results = {}
    for model_name, model in models.items():
        # Cross-validation for model performance
        cv_results = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring='neg_mean_squared_error')
        results[model_name] = {
            'mean_score': np.mean(cv_results),
            'std_score': np.std(cv_results)
        }

        # Fit the model and print the best alpha for Lasso and Ridge
        model.fit(X_train_scaled, y_train)
        
        if model_name == 'Lasso':
            print(f"Lasso - Best Alpha: {model.alpha_}")
        if model_name == 'Ridge':
            print(f"Ridge - Best Alpha: {model.alpha_}")

    # Display cross-validation results
    for model_name, result in results.items():
        print(f"{model_name} - Mean CV Score: {result['mean_score']} (Std Dev: {result['std_score']})")

    # Test set evaluation and select the best model
    best_model_name = None
    best_mse = float('inf')

    for model_name, model in models.items():
        y_pred = model.predict(X_test_scaled)
        mse = np.mean((y_test - y_pred) ** 2)
        print(f"{model_name} - Test MSE: {mse}")
        
        # Update best model if current model has lower MSE
        if mse < best_mse:
            best_mse = mse
            best_model_name = model_name

    print(f"\nThe best model based on Test MSE is: {best_model_name}")


Weather Data Sample:
              STATION                                STATION_NAME      DATE  \
0  GHCND:USW00024233  SEATTLE TACOMA INTERNATIONAL AIRPORT WA US  20120101   
1  GHCND:USW00024233  SEATTLE TACOMA INTERNATIONAL AIRPORT WA US  20120102   
2  GHCND:USW00024233  SEATTLE TACOMA INTERNATIONAL AIRPORT WA US  20120103   
3  GHCND:USW00024233  SEATTLE TACOMA INTERNATIONAL AIRPORT WA US  20120104   
4  GHCND:USW00024233  SEATTLE TACOMA INTERNATIONAL AIRPORT WA US  20120105   

   PRCP  SNWD  SNOW  TMAX  TMIN  AWND  WDF2  ...  WT17  WT05  WT02  WT22  \
0     0     0     0   128    50    47   100  ... -9999 -9999 -9999 -9999   
1   109     0     0   106    28    45   180  ... -9999 -9999 -9999 -9999   
2     8     0     0   117    72    23   180  ... -9999 -9999 -9999 -9999   
3   203     0     0   122    56    47   180  ... -9999 -9999 -9999 -9999   
4    13     0     0    89    28    61   200  ... -9999 -9999 -9999 -9999   

   WT04  WT13  WT16  WT08  WT18  WT03  
0 -9999 -999

In [2]:
print(weather_data['DATE'].head())
print(fremont_data['DATE'].head())


0   2012-01-01
1   2012-01-02
2   2012-01-03
3   2012-01-04
4   2012-01-05
Name: DATE, dtype: datetime64[ns]
0   2019-01-01 00:00:00
1   2019-01-01 01:00:00
2   2019-01-01 02:00:00
3   2019-01-01 03:00:00
4   2019-01-01 04:00:00
Name: DATE, dtype: datetime64[ns]
